# Unión de las 4 bases geoespaciales — Malla Vial, Puentes, Malla Peatonal y Semáforos

Este cuaderno **importa directamente las 4 capas ya procesadas** (CRS estandarizado a `EPSG:3116` y columnas depuradas), generadas al final de:

- `EDA_Malla_Vial_Puentes_zona_urbana.ipynb` → exporta `Data/processed/malla_vial_procesada.gpkg` y `Data/processed/puentes_procesado.gpkg`
- `EDA_Malla_peatonal_semaforos.ipynb` → exporta `Data/processed/peatonal_procesada.gpkg` y `Data/processed/semaforos_procesados.gpkg`

> **Importante:** corre esos dos notebooks primero (ya tienen al final una celda de exportación agregada) para generar los 4 archivos en `Data/processed/`. Este cuaderno no vuelve a tocar los datos crudos ni repite ninguna transformación — parte de ahí directo hacia la unión.

**Enfoque de unión: unión espacial por proximidad (`sjoin_nearest`).**

Como las 4 capas tienen geometrías y esquemas de columnas distintos (líneas, polígonos, puntos), no tiene sentido un `concat` fila a fila. En su lugar:

1. Se toma la **Malla Vial Integral** como capa base (esqueleto de la red).
2. **Puentes**, **Semáforos** y **Malla Peatonal** se relacionan cada uno con el segmento vial **más cercano**, agregando el identificador del segmento vial y la distancia (en metros) al resultado.
3. Se exporta un **GeoPackage único** con las 4 capas ya enriquecidas, listo para análisis conjunto o cruces posteriores (ej. accidentalidad, semaforización por tipo de vía, etc.).

## 1. Librerías

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
plt.rcParams['figure.figsize'] = (9, 6)

## 2. Importar las 4 bases ya procesadas

Se leen directamente desde `Data/processed/`, generadas por los dos EDA originales. No se aplica CRS, limpieza ni selección de columnas aquí — ya vienen resueltas.

In [ ]:
malla = gpd.read_file("Data/processed/malla_vial_procesada.gpkg")
puentes = gpd.read_file("Data/processed/puentes_procesado.gpkg")
peatonal = gpd.read_file("Data/processed/peatonal_procesada.gpkg")
semaforos = gpd.read_file("Data/processed/semaforos_procesados.gpkg")

for nombre, capa in [('malla', malla), ('puentes', puentes), ('peatonal', peatonal), ('semaforos', semaforos)]:
    print(f"{nombre:10s} -> {capa.shape[0]:>7,} filas | CRS: {capa.crs} | geom: {capa.geom_type.unique()}")

## 3. Base para la unión: identificador de segmento vial

Se usa `malla` (ya procesada) como capa de referencia. Se define `vial_id` a partir de `MVICODIGO` (o `OBJECTID` si no existe) para poder referenciar el segmento vial más cercano desde las otras 3 capas.

In [ ]:
id_col_malla = 'MVICODIGO' if 'MVICODIGO' in malla.columns else 'OBJECTID'
malla_base = malla.rename(columns={id_col_malla: 'vial_id'}).copy()

cols_vial_ref = [c for c in ['vial_id', 'MVITIPO', 'MVISVIA', 'longitud_m', 'geometry'] if c in malla_base.columns]
malla_base = malla_base[cols_vial_ref]
print(malla_base.shape)
malla_base.head()

## 4. Unión espacial por proximidad

Se usa `gpd.sjoin_nearest`, que para cada geometría de la capa izquierda encuentra la geometría más cercana de la capa derecha (`malla_base`) y añade su distancia real en metros (`dist_m`).

`sjoin_nearest` no admite geometrías nulas; se excluyen puntualmente solo para el join (sin sobrescribir `puentes`, `peatonal` ni `semaforos` ya procesados).

In [ ]:
def unir_a_malla_mas_cercana(capa, malla_base, sufijo, max_distancia=None):
    """
    Une `capa` (ya procesada, sin modificar) con el segmento vial más cercano de `malla_base`.
    Agrega columnas 'vial_id', 'MVITIPO', 'MVISVIA' (si existen) y 'dist_m'.
    """
    capa_valida = capa[~capa.geometry.isna()].copy()
    cols_vial = [c for c in ['vial_id', 'MVITIPO', 'MVISVIA'] if c in malla_base.columns] + ['geometry']
    resultado = gpd.sjoin_nearest(
        capa_valida,
        malla_base[cols_vial],
        how='left',
        distance_col='dist_m',
        lsuffix='',
        rsuffix='vial'
    )
    resultado = resultado.drop(columns=[c for c in resultado.columns if c == 'index_vial'], errors='ignore')
    if max_distancia is not None:
        n_lejos = (resultado['dist_m'] > max_distancia).sum()
        print(f"[{sufijo}] {n_lejos} de {len(resultado)} registros a más de {max_distancia} m del segmento vial más cercano")
    return resultado

In [ ]:
UMBRAL_M = 100  # metros; solo para reportar, no se filtra nada por defecto

puentes_unido = unir_a_malla_mas_cercana(puentes, malla_base, 'puentes', UMBRAL_M)
semaforos_unido = unir_a_malla_mas_cercana(semaforos, malla_base, 'semaforos', UMBRAL_M)
peatonal_unido = unir_a_malla_mas_cercana(peatonal, malla_base, 'peatonal', UMBRAL_M)

print("\nPuentes unidos   :", puentes_unido.shape)
print("Semáforos unidos :", semaforos_unido.shape)
print("Peatonal unida   :", peatonal_unido.shape)

### 4.1 Distribución de distancias al segmento vial más cercano

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

puentes_unido['dist_m'].plot(kind='hist', bins=40, ax=axes[0], color='crimson')
axes[0].set_title('Puentes -> distancia a vía (m)')

semaforos_unido['dist_m'].plot(kind='hist', bins=40, ax=axes[1], color='darkorange')
axes[1].set_title('Semáforos -> distancia a vía (m)')

peatonal_unido['dist_m'].plot(kind='hist', bins=40, ax=axes[2], color='seagreen')
axes[2].set_title('Peatonal -> distancia a vía (m)')

plt.tight_layout()
plt.show()

for nombre, capa in [('Puentes', puentes_unido), ('Semáforos', semaforos_unido), ('Peatonal', peatonal_unido)]:
    print(nombre, '-> dist_m describe:')
    print(capa['dist_m'].describe())
    print()

## 6. Vista conjunta de las 4 capas ya unidas

Recorte a la zona urbana (mismos límites usados en los EDA originales).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

malla_base.plot(ax=ax, linewidth=0.12, color='gray', zorder=1, label='Malla vial')
peatonal_unido.plot(ax=ax, linewidth=0.5, color='seagreen', zorder=2, label='Malla peatonal')
puentes_unido.plot(ax=ax, color='crimson', markersize=4, zorder=3, label='Puentes')
semaforos_unido.plot(ax=ax, color='darkorange', markersize=3, zorder=4, label='Semáforos')

ax.set_xlim(981000, 1010500)
ax.set_ylim(983500, 1027500)
ax.set_title('Malla Vial + Puentes + Malla Peatonal + Semáforos — Bogotá D.C. (zona urbana)')
ax.set_axis_off()
ax.legend(loc='lower right')
plt.show()

## 7. Exportar el resultado consolidado

Se guarda un único **GeoPackage** con 4 capas (una por dataset), cada una ya en `EPSG:3116` y con la información de unión (`vial_id`, `dist_m`) donde aplica. Esto facilita abrir todo en QGIS o volver a cargarlo en pandas/geopandas por capa.

In [ ]:
import os
os.makedirs('Data/processed', exist_ok=True)

salida = 'Data/processed/malla_urbana_unificada.gpkg'

malla_base.to_file(salida, layer='malla_vial', driver='GPKG')
puentes_unido.to_file(salida, layer='puentes', driver='GPKG')
semaforos_unido.to_file(salida, layer='semaforos', driver='GPKG')
peatonal_unido.to_file(salida, layer='peatonal', driver='GPKG')

print('Archivo generado:', salida)
print('Capas:', ['malla_vial', 'puentes', 'semaforos', 'peatonal'])

## 8. Resumen

- Las 4 capas se estandarizaron a un único CRS (`EPSG:3116`), condición necesaria para cualquier unión espacial.
- **Puentes**, **Semáforos** y **Malla Peatonal** se unieron a la **Malla Vial Integral** por proximidad (`sjoin_nearest`), añadiendo el `vial_id` del segmento más cercano y la distancia real en metros (`dist_m`).
- Se reportó cuántos registros de cada capa quedan a más de 100 m del segmento vial más cercano — útil para detectar posibles errores de digitalización o zonas peatonales/puentes no directamente asociadas a la malla vial modelada.
- El resultado se consolidó en un único GeoPackage (`Data/processed/malla_urbana_unificada.gpkg`) con 4 capas, listo para análisis conjunto o para cargar en un SIG.

**Siguientes pasos sugeridos:** cruzar `vial_id` entre capas para, por ejemplo, contar semáforos por tipo de vía (`MVITIPO`), o analizar la cobertura de malla peatonal por segmento vial.